In [133]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_MODEL_VECTOR_DIMENSIONS = 3072
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [134]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Return cosine similarities between a single query vector and a 3D matrix
    of shape (num_indices, size, embedding_dim).
    Zero-padded rows remain zero in the output.
    """
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  


### Get Candidate Info

In [135]:
RESUME_ID = "b6e8165a-b1af-4117-8a29-4a3a5fc95f32"
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

### Get Matching Market Info

In [136]:
matching_jobs_df = filter_job_postings(candidate_industries)
matching_jobs_ids = matching_jobs_df["id"].to_list()

market_hard_skills_df = get_position_skills(matching_jobs_ids, HARD_SKILLS_TABLE)
market_soft_skills_df = get_position_skills(matching_jobs_ids, SOFT_SKILLS_TABLE)

market_soft_skills_df = market_soft_skills_df.sort_values("job_id")
market_soft_skills_df["index"] = market_soft_skills_df.groupby("job_id").ngroup()

market_hard_skills_df = market_hard_skills_df.sort_values("job_id")
market_hard_skills_df["index"] = market_hard_skills_df.groupby("job_id").ngroup()

# STRING
### Jobs Skills Matrixes

In [137]:
string_hard_skills_df = market_hard_skills_df[["index", "skill_description"]]
string_soft_skills_df = market_soft_skills_df[["index", "skill_description"]]

hard_skills_size = string_hard_skills_df.groupby('index').size().max()
soft_skills_size = string_soft_skills_df.groupby('index').size().max()

string_hard_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in string_hard_skills_df.groupby('index')
])

string_soft_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in string_soft_skills_df.groupby('index')
])

# VECTOR
### Jobs Skills Matrixes

In [138]:
vector_hard_skills_df = market_hard_skills_df[["index", "embedding"]]
vector_soft_skills_df = market_soft_skills_df[["index", "embedding"]]

vector_hard_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (hard_skills_size - len(group)))
    for _, group in vector_hard_skills_df.groupby('index')
], dtype=np.float32)

vector_soft_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (soft_skills_size - len(group)))
    for _, group in vector_soft_skills_df.groupby('index')
], dtype=np.float32)

### Find market best matches

For each soft skill

In [ ]:
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.69
SOFT_SKILLS_EMBEDDING_COLUMN_INDEX = 2
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_soft_skills_df.shape[0] - 1

for i in range(0, skills_count):
    vector_position = (i, SOFT_SKILLS_STRING_COLUMN_INDEX) 
    string_position = (i, SOFT_SKILLS_EMBEDDING_COLUMN_INDEX)
    weight_position = (i, SOFT_SKILLS_WEIGHT_COLUMN_INDEX)

    skill_embedding = candidate_soft_skills_df.iloc[vector_position] 
    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_soft_skills_matrix)
    binary_mask = (cosine_similarities > SOFT_SKILLS_SIMILARITY_THRESHOLD).astype(np.int8)

    matched_skills_matrix = np.where(binary_mask == 1, string_soft_skills_matrix, "")
    flat_soft_skills = matched_skills_matrix[matched_skills_matrix != ""].flatten()
    
    skill_match_weight = len(flat_soft_skills)
    print(f"Skill: {candidate_soft_skills_df.iloc[string_position]}\nUnique matching values for skill are: {set(flat_soft_skills)}\nWeight: {skill_match_weight} for the set\n")


For each hard skill

In [ ]:
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.74
HARD_SKILLS_EMBEDDING_COLUMN_INDEX = 2
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_hard_skills_df.shape[0] - 1

for i in range(0, skills_count):
    vector_position = (i, HARD_SKILLS_STRING_COLUMN_INDEX) 
    string_position = (i, HARD_SKILLS_EMBEDDING_COLUMN_INDEX)
    weight_position = (i, HARD_SKILLS_WEIGHT_COLUMN_INDEX)

    skill_embedding = candidate_hard_skills_df.iloc[vector_position] 
    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_hard_skills_matrix)
    binary_mask = (cosine_similarities > HARD_SKILLS_SIMILARITY_THRESHOLD).astype(np.int8)

    matched_skills_matrix = np.where(binary_mask == 1, string_hard_skills_matrix, "")
    flat_hard_skills = matched_skills_matrix[matched_skills_matrix != ""].flatten()

    skill_match_weight = len(flat_hard_skills)
    print(f"Skill: {candidate_hard_skills_df.iloc[string_position]}\nUnique matching values for skill are: {set(flat_hard_skills)}\nWeight: {skill_match_weight} for the set\n")

### Find candidate's missing skills for market 